# Clase 215 — Star schema completo con DuckDB

Construimos un star schema de e-commerce: `dim_date`, `dim_customer` (SCD 2), `dim_product`, `dim_store`, `fact_sales`. Demostramos SCD 2 con cliente que cambia de ciudad.

In [ ]:
import duckdb, tempfile
from pathlib import Path
DB = str(Path(tempfile.gettempdir()) / 'dw_star.duckdb')
Path(DB).unlink(missing_ok=True)
con = duckdb.connect(DB)

## 1. `dim_date` — 5 años de fechas precalculadas

In [ ]:
con.execute('''
    CREATE TABLE dim_date AS
    SELECT
        CAST(strftime(d, '%Y%m%d') AS INT) AS date_key,
        d                                  AS date,
        EXTRACT('year'  FROM d)            AS year,
        EXTRACT('quarter' FROM d)          AS quarter,
        EXTRACT('month' FROM d)            AS month,
        strftime(d, '%B')                  AS month_name,
        EXTRACT('day'   FROM d)            AS day,
        EXTRACT('isodow' FROM d)           AS day_of_week_iso,
        strftime(d, '%A')                  AS day_name,
        CASE WHEN EXTRACT('isodow' FROM d) IN (6, 7) THEN TRUE ELSE FALSE END AS is_weekend,
        CASE WHEN EXTRACT('month' FROM d) >= 10 THEN EXTRACT('year' FROM d) + 1
             ELSE EXTRACT('year' FROM d) END AS fiscal_year
    FROM (SELECT UNNEST(generate_series(DATE '2022-01-01', DATE '2026-12-31', INTERVAL 1 DAY)) AS d)
''')
print(con.execute('SELECT COUNT(*), MIN(date), MAX(date) FROM dim_date').fetchone())
print(con.execute("SELECT * FROM dim_date WHERE date='2026-06-17'").fetchdf())

## 2. `dim_product` — denormalizada

In [ ]:
con.execute('''
    CREATE TABLE dim_product AS
    SELECT
        ROW_NUMBER() OVER () AS product_key,         -- surrogate key
        sku,                                          -- natural key
        name, category, brand, list_price
    FROM (VALUES
        ('SKU-001', 'Coffee Mug',   'Kitchen',  'AcmeBrand',  12.0),
        ('SKU-002', 'T-Shirt',      'Apparel',  'BlueLabel',  25.0),
        ('SKU-003', 'Notebook',     'Office',   'NoteCo',      8.0),
        ('SKU-004', 'Headphones',   'Electronics', 'AudioPro', 150.0),
        ('SKU-005', 'Water Bottle', 'Kitchen',  'AcmeBrand',  18.0)
    ) v(sku, name, category, brand, list_price)
''')
print(con.execute('SELECT * FROM dim_product').fetchdf())

## 3. `dim_customer` — SCD Tipo 2

In [ ]:
con.execute('''
    CREATE TABLE dim_customer (
        customer_key INT PRIMARY KEY,
        customer_id  TEXT,            -- natural key
        name TEXT, email TEXT,
        city TEXT, country TEXT,
        valid_from DATE NOT NULL,
        valid_to   DATE,              -- NULL = vigente
        is_current BOOLEAN NOT NULL
    )
''')

# Carga inicial (estado al 2024-01-01)
con.execute("""
    INSERT INTO dim_customer VALUES
        (1, 'C001', 'Alice', 'a@x.com', 'Buenos Aires', 'AR', DATE '2024-01-01', NULL, TRUE),
        (2, 'C002', 'Bob',   'b@x.com', 'Montevideo',   'UY', DATE '2024-01-01', NULL, TRUE),
        (3, 'C003', 'Carol', 'c@x.com', 'Santiago',     'CL', DATE '2024-01-01', NULL, TRUE)
""")
print(con.execute('SELECT * FROM dim_customer').fetchdf())

## 4. `fact_sales` — grain: 1 row per order line

In [ ]:
con.execute('''
    CREATE TABLE fact_sales (
        sale_id      BIGINT,
        date_key     INT REFERENCES dim_date,
        product_key  INT REFERENCES dim_product,
        customer_key INT REFERENCES dim_customer,
        qty INT, revenue DOUBLE, discount DOUBLE
    )
''')

# 100 ventas sintéticas
con.execute("""
    INSERT INTO fact_sales
    SELECT
        i AS sale_id,
        CAST(strftime(DATE '2024-06-01' + INTERVAL ((random()*100)::INT) DAY, '%Y%m%d') AS INT) AS date_key,
        (1 + (random() * 4)::INT) AS product_key,
        (1 + (random() * 2)::INT) AS customer_key,
        (1 + (random() * 5)::INT) AS qty,
        20 + random() * 100       AS revenue,
        random() * 5              AS discount
    FROM range(100) t(i)
""")
print(con.execute('SELECT COUNT(*), SUM(revenue) FROM fact_sales').fetchone())

## 5. SCD 2 en acción: Alice se muda Buenos Aires → Madrid

In [ ]:
# Pre-move: ventas de Alice se asocian a Buenos Aires
pre = con.execute("""
    SELECT c.city, COUNT(*) sales, SUM(f.revenue) rev
    FROM fact_sales f
    JOIN dim_customer c ON f.customer_key = c.customer_key
    WHERE c.customer_id = 'C001'
    GROUP BY c.city
""").fetchdf()
print('Antes del cambio:')
print(pre)

# SCD 2 update: cerrar fila vigente, insertar nueva
con.execute("""
    UPDATE dim_customer
    SET valid_to = DATE '2024-09-01', is_current = FALSE
    WHERE customer_id = 'C001' AND is_current = TRUE
""")
con.execute("""
    INSERT INTO dim_customer VALUES (4, 'C001', 'Alice', 'a@x.com', 'Madrid', 'ES', DATE '2024-09-02', NULL, TRUE)
""")

# Nuevas ventas posteriores referencian customer_key=4 (Madrid)
con.execute("""
    INSERT INTO fact_sales VALUES
        (1001, 20240910, 1, 4, 2, 50.0, 0),
        (1002, 20240915, 2, 4, 1, 25.0, 0)
""")

post = con.execute("""
    SELECT c.city, COUNT(*) sales, SUM(f.revenue) rev
    FROM fact_sales f
    JOIN dim_customer c ON f.customer_key = c.customer_key
    WHERE c.customer_id = 'C001'
    GROUP BY c.city
    ORDER BY rev DESC
""").fetchdf()
print('\nDespués del cambio (ventas correctamente atribuidas a la ciudad de cada momento):')
print(post)

## 6. Query analítica clásica usando todas las dims

In [ ]:
result = con.execute('''
    SELECT
        p.brand,
        d.fiscal_year,
        d.is_weekend,
        COUNT(*)            AS n_orders,
        SUM(f.revenue)      AS revenue,
        AVG(f.revenue)      AS avg_order
    FROM fact_sales f
    JOIN dim_product  p USING (product_key)
    JOIN dim_date     d USING (date_key)
    GROUP BY p.brand, d.fiscal_year, d.is_weekend
    ORDER BY revenue DESC
    LIMIT 10
''').fetchdf()
print(result)

In [ ]:
con.close(); print('done.')

## Ejercicio guiado

1. Definí el grain de tu fact table para un caso real propio. Justificá.
2. Agregá una `dim_store` con jerarquía geográfica (store → city → country). Decidí star (denormalizada) o snowflake (separar `dim_city` y `dim_country`).
3. Implementá la query: cohort retention mensual usando `dim_date` + `fact_sales`.
4. Convertí el script a dbt models: `models/staging/`, `models/marts/dim_*.sql`, `models/marts/fact_*.sql`. Agregá `tests` (unique, not_null, accepted_values).
5. Bonus: mismo schema en BigQuery con `PARTITION BY date_key` + `CLUSTER BY product_key, customer_key`. Compará costo de queries.

## Conclusiones

- Star schema sigue siendo el patrón ganador en 2026 para data warehouses.
- Grain definido al inicio = decisión que evita los bugs más caros.
- SCD 2 con surrogate keys = único modo correcto de manejar history de entidades.
- `dim_date` precalculada > derivar en query.
- **Fin de Parte 5**: tenés orquestación (208-209) + procesamiento (210-211) + DW (212) + streaming (213) + formatos (214) + modelado (215) = stack completo para alimentar ML a escala.